# BIT Training Test Notebook

This notebook demonstrates how to:
1. Load neural decoding datasets
2. Instantiate a BIT model using AutoModel
3. Wrap the model in the BITTrainer PyTorch Lightning module
4. Launch training with validation

The trainer uses CTC loss and computes per-dataset Character Error Rate (CER) metrics.

In [2]:
import sys
import os
import pickle
from pathlib import Path

import torch
from torch.utils.data import DataLoader, ConcatDataset
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from transformers import AutoConfig, AutoModel
from neural_fm.models.speech_decoding.bit_trainer import BITTrainer
from neural_fm.models.speech_decoding.bit_cross_encoder_config import BitCrossEncoderConfig
from neural_fm.models.speech_decoding.bit_cross_encoder_model import BitCrossEncoderModel
# Add project paths to sys.path
# project_root = Path("/data/data/matteo/neural_foundation_model_invasive")
# sys.path.insert(0, str(project_root / "src" / "neural_fm" / "models" / "speech_decoding"))
# sys.path.insert(0, str(project_root / "src" / "neural_fm"))


# Import model classes
# from bit_cross_encoder_config import BitCrossEncoderConfig
# from bit_cross_encoder_model import BitCrossEncoderModel
# from bit_trainer import BITTrainer

# Import dataset utilities
# Note: These imports assume the dataset utilities are available
# You may need to adjust the import paths based on your project structure
from neural_fm.config import *
from neural_fm.neural_datasets.datasets import load_all_datasets, collate_fn_flexible
# Import config for dataset paths
# from config import (
#     TRAIN_CARD_DATASET, 
#     VAL_CARD_DATASET, 
#     TEST_CARD_DATASET,
#     TRAIN_WILLET_DATASET, 
#     TEST_WILLET_DATASET,
#     TRAIN_HANDWRITING_DATASET,
#     TEST_HANDWRITING_DATASET
# )

print("All imports successful!")

All imports successful!


## Configuration

Set up training hyperparameters and paths.

In [6]:
# Training configuration
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_EPOCHS = 50
MAX_STEPS = 120000
LEARNING_RATE = 1e-3
LR_MIN = 1e-3
WEIGHT_DECAY = 1e-5
LR_WARMUP_STEPS = 1000
LR_DECAY_STEPS = 120000

# Data augmentation
WHITE_NOISE_SD = 0.2
CONSTANT_OFFSET_SD = 0.05

# Checkpoint directory
CHECKPOINT_DIR =  "checkpoints/bit_training"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# Wandb logging (set to None to disable)
WANDB_PROJECT = "bit-neural-decoding"
WANDB_ENTITY = None  # Set to your wandb entity if needed

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")

Checkpoint directory: checkpoints/bit_training
Batch size: 64
Max epochs: 50
Learning rate: 0.001


## Load Datasets

Load the neural decoding datasets from pickle files.

In [7]:
datasets = load_all_datasets()

Loading kunz datasets
loading card dataset
loading willett dataset
loading handwriting (Willett 2021) dataset
loading wairagkar dataset
loading fan handwriting dataset
loading neuralpile dataset


## Create PyTorch Datasets

Wrap the loaded data in PyTorch Dataset objects.

In [9]:
# Create PyTorch datasets
print("Creating PyTorch datasets...")

train_card_dataset = datasets["card"]["train"]
val_card_dataset = datasets["card"]["val"]
test_card_dataset = datasets["card"]["test"]

train_willet_dataset = datasets["willet"]["train"]
val_willet_dataset = datasets["willet"]["val"]
test_willet_dataset = datasets["willet"]["test"]

# Combine training datasets
combined_train_dataset = ConcatDataset([
    train_card_dataset,
    train_willet_dataset,
    val_willet_dataset,
])

# Use test_card_dataset for validation
val_dataset = val_card_dataset

print(f"\nDataset sizes:")
print(f"  Train Card: {len(train_card_dataset)}")
print(f"  Val Card: {len(val_card_dataset)}")
print(f"  Test Card: {len(test_card_dataset)}")
print(f"  Train Willet: {len(train_willet_dataset)}")
print(f"  Test Willet: {len(test_willet_dataset)}")
print(f"\nCombined training set size: {len(combined_train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

Creating PyTorch datasets...

Dataset sizes:
  Train Card: 8069
  Val Card: 1426
  Test Card: 1450
  Train Willet: 8800
  Test Willet: 1200

Combined training set size: 17749
Validation set size: 1426


## Create DataLoaders

Create PyTorch DataLoaders for training and validation.

In [10]:
# Create DataLoaders
train_loader = DataLoader(
    combined_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn_flexible,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_flexible,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

# Test batch loading
print("\nTesting batch loading...")
sample_batch = next(iter(train_loader))
print(f"Sample batch keys: {sample_batch.keys()}")
print(f"Neural features shape: {sample_batch['neural_features'].shape}")
print(f"Sequence class IDs shape: {sample_batch['seq_class_ids'].shape}")
print(f"N time steps shape: {sample_batch['n_time_steps'].shape}")
print(f"Sequence lengths shape: {sample_batch['seq_lengths'].shape}")
print(f"Source datasets: {sample_batch['source_dataset'][:5]}...")  # Show first 5

Train loader: 278 batches
Val loader: 23 batches

Testing batch loading...
Sample batch keys: dict_keys(['neural_features', 'neural_lengths', 'n_time_steps', 'neural_mask', 'seq_class_ids', 'seq_lengths', 'seq_mask', 'day', 'sentence_label', 'transcriptions', 'source_dataset'])
Neural features shape: torch.Size([64, 2160, 512])
Sequence class IDs shape: torch.Size([64, 58])
N time steps shape: torch.Size([64])
Sequence lengths shape: torch.Size([64])
Source datasets: ['card', 'willet', 'card', 'willet', 'card']...


## Create Model

Instantiate the BIT model using the HuggingFace configuration and AutoModel.

In [12]:
# Register model for AutoModel usage
AutoConfig.register("bit_cross_encoder", BitCrossEncoderConfig)
AutoModel.register(BitCrossEncoderConfig, BitCrossEncoderModel)

# Create model configuration
# Note: patch_size width must match the feature dimension of your data
config = BitCrossEncoderConfig(
    patch_size=(5, 512),  # Adjust patch_width (512) to match your data's feature dimension
    dim=784,
    depth=7,
    heads=4,
    mlp_dim_ratio=4,
    dim_head=64,
    dropout=0.4,
    input_dropout=0.2,
    gaussian_smooth_width=2.0,
    n_classes=40,  # Number of phoneme classes (excluding blank)
    t5_style_pos=True,
    max_mask_pct=0.075,
    num_masks=20,
    mask_token_zeros=False,
    num_masks_channels=0,
    max_mask_channels=0,
    dataset_keys=["card", "willet"],
    per_dataset_heads=False,
)

# Create model using AutoModel
print("Creating model...")
model = AutoModel.from_config(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model config: {config}")

Creating model...

Model created successfully!
Total parameters: 44,163,458
Trainable parameters: 44,163,458
Model config: BitCrossEncoderConfig {
  "dataset_keys": [
    "card",
    "willet"
  ],
  "depth": 7,
  "dim": 784,
  "dim_head": 64,
  "dropout": 0.4,
  "gaussian_smooth_width": 2.0,
  "heads": 4,
  "input_dropout": 0.2,
  "mask_token_zeros": false,
  "max_mask_channels": 0,
  "max_mask_pct": 0.075,
  "mlp_dim_ratio": 4,
  "model_type": "bit_cross_encoder",
  "n_classes": 40,
  "num_masks": 20,
  "num_masks_channels": 0,
  "patch_size": [
    5,
    512
  ],
  "per_dataset_heads": false,
  "t5_style_pos": true,
  "transformers_version": "4.57.2"
}



## Create PyTorch Lightning Trainer Wrapper

Wrap the model in the BITTrainer PyTorch Lightning module.

In [13]:
# Create PyTorch Lightning wrapper
print("Creating BITTrainer wrapper...")
lightning_model = BITTrainer(
    model=model,
    learning_rate=LEARNING_RATE,
    lr_min=LR_MIN,
    lr_warmup_steps=LR_WARMUP_STEPS,
    lr_decay_steps=LR_DECAY_STEPS,
    weight_decay=WEIGHT_DECAY,
    white_noise_sd=WHITE_NOISE_SD,
    constant_offset_sd=CONSTANT_OFFSET_SD,
    beta0=0.9,
    beta1=0.999,
    epsilon=1e-8,
    # Optional: override masking parameters
    # max_mask_pct_override=0.05,
    # num_masks_override=10,
)

print("BITTrainer wrapper created successfully!")

Creating BITTrainer wrapper...
BITTrainer wrapper created successfully!


## Setup Training Callbacks

Configure callbacks for model checkpointing and early stopping.

In [14]:
# Setup callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="bit-{epoch:02d}-{val_CER_card:.4f}",
    monitor="val_CER_card", #possibly change to cermetric
    mode="min",
    save_top_k=3,
    save_last=True,
    verbose=True,
)

# early_stop_callback = EarlyStopping(
#     monitor="val_loss",
#     patience=50,
#     mode="min",
#     verbose=True,
# )

callbacks = [checkpoint_callback]

print(f"Callbacks configured:")
print(f"  - Model checkpointing (save top 3, monitor: val_loss)")
print(f"  - Early stopping (patience: 10, monitor: val_loss)")

Callbacks configured:
  - Model checkpointing (save top 3, monitor: val_loss)
  - Early stopping (patience: 10, monitor: val_loss)


## Setup Logger (Optional)

Configure Weights & Biases logger for experiment tracking.

In [15]:
# Setup logger (optional)
logger = None
if WANDB_PROJECT:
    try:
        import wandb
        logger = WandbLogger(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=f"bit-training-{torch.cuda.device_count()}gpu",
            log_model=False,
        )
        print(f"Wandb logger configured: {WANDB_PROJECT}")
    except ImportError:
        print("Wandb not available, skipping logger setup")
        logger = None
else:
    print("No logger configured (set WANDB_PROJECT to enable)")

Wandb logger configured: bit-neural-decoding


## Launch Training

Create the PyTorch Lightning Trainer and start training.

**Note:** Mixed precision (16-bit) training is now supported and recommended for faster training on modern GPUs.

In [16]:
# Create PyTorch Lightning Trainer
# Mixed precision (16-bit) is now supported and recommended for faster training
trainer = pl.Trainer(
    max_steps=MAX_STEPS,
    accelerator="auto",
    devices="auto",
    callbacks=callbacks,
    logger=logger,
    gradient_clip_val=2.0,
    accumulate_grad_batches=1,
    # precision="16-mixed",  # Use mixed precision for faster training (recommended)
    val_check_interval=1.0,  # Validate once per epoch
    enable_progress_bar=True,
)

print(f"\nTrainer configured:")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Accelerator: auto")
print(f"  Devices: auto")
print(f"  Gradient clipping: 1.0")
print(f"  Precision: 16-mixed (faster training with automatic mixed precision)")
print(f"\nStarting training...\n")

# Train the model
trainer.fit(
    lightning_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("\nTraining completed!")

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
/home/matteo/anaconda3/envs/b2txt25_ens/lib/python3.10/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf


Trainer configured:
  Max epochs: 50
  Accelerator: auto
  Devices: auto
  Gradient clipping: 1.0
  Precision: 16-mixed (faster training with automatic mixed precision)

Starting training...



wandb: Currently logged in as: matteoferrante to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name     | Type                 | Params | Mode 
----------------------------------------------------------
0 | model    | BitCrossEncoderModel | 44.2 M | train
1 | ctc_loss | CTCLoss              | 0      | train
----------------------------------------------------------
44.2 M    Trainable params
0         Non-trainable params
44.2 M    Total params
176.654   Total estimated model params size (MB)
144       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 429, global step 119540: 'val_CER_card' was not in top 3


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 430, global step 119818: 'val_CER_card' was not in top 3
Epoch 431, global step 120000: 'val_CER_card' was not in top 3
`Trainer.fit` stopped: `max_steps=120000` reached.



Training completed!


## Training Results

Display training results and best checkpoint information.

In [ ]:
# Display results
print("\n" + "="*80)
print("TRAINING RESULTS")
print("="*80)

if hasattr(checkpoint_callback, 'best_model_path'):
    print(f"\nBest model path: {checkpoint_callback.best_model_path}")
    print(f"Best model score: {checkpoint_callback.best_model_score:.4f}")

print(f"\nCheckpoints saved to: {CHECKPOINT_DIR}")
print(f"\nTo load the best model:")
print(f"  checkpoint_path = '{checkpoint_callback.best_model_path}'")
print(f"  loaded_model = BITTrainer.load_from_checkpoint(checkpoint_path, model=model)")


TRAINING RESULTS

Best model path: /data/data/matteo/neural_foundation_model_invasive/checkpoints/bit_training/bit-epoch=12-val_loss=0.6468.ckpt
Best model score: 0.6468

Checkpoints saved to: /data/data/matteo/neural_foundation_model_invasive/checkpoints/bit_training

To load the best model:
  checkpoint_path = '/data/data/matteo/neural_foundation_model_invasive/checkpoints/bit_training/bit-epoch=12-val_loss=0.6468.ckpt'
  loaded_model = BITTrainer.load_from_checkpoint(checkpoint_path, model=model)


## Test Inference (Optional)

Test the trained model on a sample batch.

In [ ]:
# Load best model for inference
print("Loading best model for inference...")
best_model = BITTrainer.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    model=model,
)
best_model.eval()
best_model = best_model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Get a validation batch
val_batch = next(iter(val_loader))

# Run inference
print("\nRunning inference on sample batch...")
with torch.no_grad():
    X = val_batch["neural_features"].to(best_model.device)
    X_len = val_batch["n_time_steps"].to(best_model.device)
    
    # Forward pass
    logits = best_model(X, X_len)
    
    print(f"Input shape: {X.shape}")
    print(f"Output logits shape: {logits.shape}")
    
    # Decode predictions
    predictions = torch.argmax(logits, dim=-1)
    print(f"Predictions shape: {predictions.shape}")
    print(f"Sample prediction (first sequence): {predictions[0, :20].cpu().numpy()}")

print("\nInference test completed!")

Loading best model for inference...

Running inference on sample batch...
Input shape: torch.Size([32, 1203, 512])
Output logits shape: torch.Size([32, 241, 41])
Predictions shape: torch.Size([32, 241])
Sample prediction (first sequence): [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 37 37]

Inference test completed!


## Save Model to HuggingFace Format (Optional)

Save the trained model in HuggingFace format for easy loading with AutoModel.

In [ ]:
# Save model in HuggingFace format
save_path = project_root / "models" / "bit_trained_model"
save_path.mkdir(parents=True, exist_ok=True)

print(f"Saving model to {save_path}...")
best_model.model.save_pretrained(save_path)
config.save_pretrained(save_path)

print("\nModel saved successfully!")
print(f"\nTo load the model later:")
print(f"  from transformers import AutoModel")
print(f"  model = AutoModel.from_pretrained('{save_path}')")

Saving model to /data/data/matteo/neural_foundation_model_invasive/models/bit_trained_model...

Model saved successfully!

To load the model later:
  from transformers import AutoModel
  model = AutoModel.from_pretrained('/data/data/matteo/neural_foundation_model_invasive/models/bit_trained_model')
